In [ ]:
from keras.layers import Dense
from keras.models import Sequential
import matplotlib.pyplot as plt 
import numpy as np

In [ ]:
seed=7
np.random.seed(seed)

In [ ]:
dataset=np.loadtxt('pima-indians-diabetes.data.txt', delimiter=',')

In [ ]:
x=dataset[:,0:8]
y=dataset[:,8]

In [ ]:
model=Sequential()
model.add(Dense(12, input_dim=8, init='uniform', activation='relu'))
model.add(Dense(8, init='uniform', activation='relu'))
model.add(Dense(1, init='uniform', activation='sigmoid'))

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
history=model.fit(x,y, validation_split=0.33, nb_epoch=150, batch_size=10, verbose=0)

In [ ]:
print(history.history.keys())

In [ ]:
plt.plot(history.history['acc'])
plt.plot(history.history['val_loss'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('loss of the model')
plt.xlabel('nb_epoch')
plt.ylabel('loss')
plt.legend(['train','test'], loc='upper left')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from keras.layers import Dense
from keras.models import Sequential
from keras.layers import Dropout
from keras.wrappers.scikit_learn import KerasClassifier
from keras.constraints import maxnorm
from keras.optimizers import SGD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

In [ ]:
seed=7
np.random.seed(seed)

In [ ]:
dataframe=pd.read_csv('sonar.csv', header=None)

In [ ]:
dataset=dataframe.values

In [ ]:
x=dataset[:,0:60]
y=dataset[:,60]

In [ ]:
encoder=LabelEncoder()
encoder.fit(y)
encoded_y=encoder.transform(y)

In [ ]:
def baseline_model():
    model=Sequential()
    #model.add(Dropout(0.2, input_shape=(60,)))
    model.add(Dense(60, input_dim=60, init='normal', activation='relu', W_constraint=maxnorm(3)))
    model.add(Dropout(0.2))
    model.add(Dense(30, init='normal', activation='relu', W_constraint=maxnorm(3)))
    model.add(Dropout(0.2))
    model.add(Dense(1, init='normal', activation='sigmoid'))
    #compile model
    sgd=SGD(lr=0.1, momentum=0.9, decay=0)
    model.compile(loss='binary_crossentropy', optimizer='sgd', metrics=(['accuracy']))
    return model

In [ ]:
#fit and evaluate the model
estimators=[]
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasClassifier(build_fn=baseline_model, nb_epoch=300, batch_size=16, verbose=0)))
pipeline=Pipeline(estimators)
kfold=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
results=cross_val_score(pipeline, x,y, cv=kfold)

In [ ]:
results.mean()*100, results.std()*100

In [ ]:
a=np.array([[1,0,0],[0,1,0],[0,0,1]])
a.shape

In [ ]:
from sklearn.decomposition import PCA
pca=PCA(n_components=2)
pca.fit_transform(a)
#fit.components_.shape


*using learning rate schedules to fine tune the neural networks*

In [ ]:
import pandas as pd
import numpy as np
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from keras.optimizers import SGD

In [ ]:
#fix randam seed for reproducability
seed=7
np.random.seed(seed)

In [ ]:
dataframe=pd.read_csv('ionosphere.csv', delimiter=',', header=None, index_col=False)
dataset=dataframe.values
dataset.shape

In [ ]:
x=dataset[:,0:34]
y=dataset[:, 34]

In [ ]:
#one hot encoding, encode the output as integers
encoder=LabelEncoder()
encoder_y=encoder.fit(y)
encoded_y=encoder.transform(y)

In [ ]:
#create model
model=Sequential()
model.add(Dense(34, input_dim=34, init='normal', activation='relu'))
model.add(Dense(1, init='normal', activation='sigmoid'))
#compile model
learning_rate=0.1
epochs=50
momentum=0.8
decay=learning_rate/epochs
sgd=SGD(lr=learning_rate, momentum=momentum, decay=decay, nesterov=False)
model.compile(loss='binary_crossentropy', optimizer=sgd, metrics=['accuracy'])
#fit the model
model.fit(x, encoded_y, validation_split=0.33, nb_epoch=50, batch_size=28, verbose=2)

In [ ]:
#drop based learning rate decay
#defining a function to return learning rate
def step_decay(epoch):
    init_lrate=0.1
    drop=0.5
    epoch_drop=10
    lrate = initial_lrate * math.pow(drop, math.floor((1+epoch)/epochs_drop)) 
    return lrate



In [ ]:
#load the mnist dataset
from keras.datasets import mnist
import matplotlib.pyplot as plt
(x_train, y_train), (x_test, y_test)= mnist.load_data()
#plot 
plt.subplot(221)
plt.imshow(x_train[0], cmap=plt.get_cmap('gray'))
plt.subplot(222)
plt.imshow(x_train[1], cmap=plt.get_cmap('gray'))
plt.subplot(223)
plt.imshow(x_train[2], cmap=plt.get_cmap('gray'))
plt.subplot(224)
plt.imshow(x_train[3], cmap=plt.get_cmap('gray'))
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.datasets import mnist
from keras.utils import np_utils

In [ ]:
seed=7
np.random.seed(seed)

In [ ]:
(x_train, y_train),(x_test, y_test)= mnist.load_data()

In [ ]:
num_pixels=x_train.shape[1]*x_train.shape[2]

In [ ]:
x_train=x_train.reshape(x_train.shape[0], num_pixels).astype('float32')
x_test=x_test.reshape(x_test.shape[0], num_pixels).astype('float32')

In [ ]:
x_train=x_train/255
x_test=x_test/255

In [ ]:
y_train=np_utils.to_categorical(y_train)
y_test=np_utils.to_categorical(y_test)


In [ ]:
num_classes=y_test.shape[1]

In [ ]:
def baseline_model():
    model=Sequential()
    model.add(Dense(num_pixels, input_dim=num_pixels, init='normal', activation='relu'))
    model.add(Dense(num_classes, init='normal', activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [ ]:
model=baseline_model()
model.fit(x_train, y_train, validation_data=(x_test, y_test),nb_epoch=10, batch_size=200, verbose=2)

In [ ]:
scores=model.evaluate(x_test, y_test, verbose=0)
print(scores)
error_rate=100-(scores[1]*100)
print(error_rate)

In [ ]:
#building cnns
import pandas as pd
import numpy as np
from keras.layers import Dense, Dropout
from  keras.models import Sequential
from keras.datasets import mnist
from keras.layers import Flatten
from keras.layers.convolutional import Convolution2D
from keras.layers.convolutional import MaxPooling2D
from keras.utils import np_utils
from keras import backend as K
K.set_image_dim_ordering('th')

In [ ]:
seed=7
np.random.seed(seed)

In [ ]:
(x_train, y_train), (x_test, y_test)= mnist.load_data()
x_train=x_train.reshape(x_train.shape[0], 1, 28, 28).astype('float32')
x_test=x_test.reshape(x_test.shape[0], 1, 28, 28).astype('float32')
x_train.shape

In [ ]:
x_train=x_train/255
x_test=x_test/255
y_train=np_utils.to_categorical(y_train)
y_test=np_utils.to_categorical(y_test)
num_classes=y_train.shape[1]

In [ ]:
def baseline_model():
    model=Sequential()
    model.add(Convolution2D(32, (5, 5), padding='valid', strides=2, input_shape=(1, 28, 28), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [ ]:
model=baseline_model()
model.fit(x_train, y_train, validation_data=(x_test, y_test), nb_epoch=10, batch_size=200, verbose=2)

In [ ]:
scores=model.evaluate(x_test, y_test, verbose=0)
print(scores)

In [ ]:
datagen=ImageDataGenerator()